# Rigorous Baseline Benchmark

Compares **fastapriori** against academic and industry baselines using fair, kernel-measured peak RSS:

| Method | Language | Source |
|---|---|---|
| `fastapriori_fast` | Python/Rust | this package, `algo="fast"` |
| `fastapriori_classic` | Python/Rust | this package, `algo="classic"` |
| `borgelt_apriori` | C | [borgelt.net](https://borgelt.net/apriori.html) |
| `borgelt_fpgrowth` | C | [borgelt.net](https://borgelt.net/fpgrowth.html) |
| `borgelt_eclat` | C | [borgelt.net](https://borgelt.net/eclat.html) |
| `spmf_fpgrowth` | Java | [SPMF](https://www.philippe-fournier-viger.com/spmf/) |
| `spmf_eclat` | Java | SPMF |
| `mlxtend_apriori` | Python | mlxtend |
| `mlxtend_fpgrowth` | Python | mlxtend |
| `efficient_apriori` | Python | efficient-apriori |

**Methodology** — every run spawns a fresh subprocess wrapped by `/usr/bin/time -v`, so peak RSS is measured by the kernel and is comparable across languages. Runs are tiered (10/5/3) by workload size, the first run of each tier is discarded as warm-up, and both parametric (mean+std) and non-parametric (median+IQR) summaries are saved so any plot can be rebuilt from the raw CSV without re-running.

**Platform** — this notebook requires Linux (or WSL). It will not run on Windows native because `/usr/bin/time`, `gcc`, `make`, and `java` are needed.

## Section 0 — Environment probe and config

In [1]:
import os, sys, platform, shutil
from pathlib import Path

# Import helpers. baseline_runners.py lives alongside this notebook.
sys.path.insert(0, str(Path().resolve()))
import baseline_runners as br

print(f"Platform: {platform.system()} {platform.release()}")
print(f"Python:   {sys.version.split()[0]}")
print(f"CWD:      {Path().resolve()}")
print()

if platform.system() != "Linux":
    print("\u26a0\ufe0f  This notebook requires Linux or WSL.")
    print("   On Windows with WSL:  wsl -d Ubuntu   then relaunch Jupyter from WSL.")

tools = br.detect_tools()
print("Detected tools:")
for k, v in tools.items():
    mark = "\u2705" if v else "\u274c"
    print(f"  {mark} {k}: {v}")

env = br.collect_environment()
print()
print("Environment fingerprint:")
for k, v in env.items():
    print(f"  {k}: {v}")

Platform: Linux 6.17.0-1012-gcp
Python:   3.12.3
CWD:      /home/mrigank_swet/fastapriori/benchmarks

Detected tools:
  ✅ usr_bin_time: True
  ✅ gcc: True
  ✅ make: True
  ✅ java: True
  ✅ wget: True
  ✅ borgelt_apriori: True
  ✅ borgelt_fpgrowth: True
  ✅ borgelt_eclat: True
  ✅ spmf_jar: True
  ✅ mlxtend: True
  ✅ efficient_apriori: True
  ✅ fastapriori: True
  ✅ pyfim: True
  ✅ pyeclat: True
  ✅ spmf_py: True

Environment fingerprint:
  git_sha: 
  hostname: instance-20260418-153350
  cpu_model: Intel(R) Xeon(R) Platinum 8481C CPU @ 2.70GHz
  cpu_cores: 8
  ram_gb: 62.79
  python_version: 3.12.3
  rust_version: rustc 1.95.0 (59807616e 2026-04-14)
  java_version: 
  jvm_opts: 
  os_release: Linux-6.17.0-1012-gcp-x86_64-with-glibc2.39
  swappiness: 60
  cpu_governor: 
  caches_dropped: 0


In [ ]:
# Master config. Flip SMOKE=True for a laptop sanity check (~5 min); leave
# False for the full paper grid.
#
# Grid shape: per-(dataset, k) support list + per-k method list.
# Tweak any cell without touching the rest of the notebook.

SMOKE = False

# ---------------------------------------------------------------------------
# Per-(dataset, k) support thresholds. Keys with no entry are skipped silently.
# Lower supports at k=2 (lots of pair work), tighter supports at higher k where
# candidate generation explodes. Adjust individual cells as needed.
# ---------------------------------------------------------------------------
SUPPORT_GRID_FULL = {
    "Groceries": {
        2: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        3: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        4: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        5: [0.0001, 0.0005, 0.001, 0.005, 0.01]
    },
   
    "BMS-WebView-2": {
       2: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        3: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        4: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        5: [0.0001, 0.0005, 0.001, 0.005, 0.01]
    },
   
    "Retail (Belgian)": {
       2: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        3: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        4: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        5: [0.0001, 0.0005, 0.001, 0.005, 0.01]
    },
  
    "Chainstore": {
       2: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        3: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        4: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        5: [0.0001, 0.0005, 0.001, 0.005, 0.01]
    },
    "Instacart": {
        2: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        3: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        4: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        5: [0.0001, 0.0005, 0.001, 0.005, 0.01]
    },
      "Online Retail": {
       2: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        3: [ 0.0005, 0.001, 0.005, 0.01],
        4: [ 0.0005, 0.001, 0.005, 0.01],
        5: [ 0.0005, 0.001, 0.005, 0.01]
    },
    "Kosarak": {
       2: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        3: [ 0.0005, 0.001, 0.005, 0.01],
        4: [ 0.0005, 0.001, 0.005, 0.01],
        5: [ 0.0005, 0.001, 0.005, 0.01]
    },
     "BMS-WebView-1": {
        2: [0.0001, 0.0005, 0.001, 0.005, 0.01],
        3: [ 0.0005, 0.001, 0.005, 0.01],
        4: [ 0.0005, 0.001, 0.005, 0.01],
        5: [ 0.0005, 0.001, 0.005, 0.01]
    },
}

# ---------------------------------------------------------------------------
# Per-k method list. Drop Python baselines at k>2 because they're prohibitively
# slow and add no information (we already know they lose to Borgelt there).
# fastapriori_classic stays at every k as the "native Rust, textbook Apriori"
# reference that isolates the language contribution from the algorithmic one.
#
# k=2 also runs "fastapriori_fast_1t" — the same Rust binary pinned to
# RAYON_NUM_THREADS=1 by baseline_runners. That gives the paper an
# apples-to-apples single-thread row to plot alongside the default
# multi-threaded fastapriori_fast and the intrinsically-serial C/Python
# baselines (Borgelt, efficient-apriori, pyfim) on the same panel.
# ---------------------------------------------------------------------------
METHODS_BY_K_FULL = {
    2: [
        "fastapriori_fast",
        "fastapriori_fast_1t",     # same binary, RAYON_NUM_THREADS=1 (k=2 only)
        "fastapriori_classic",
        "efficient_apriori",       # Python baseline, included only at k=2
        "borgelt_apriori",
        "borgelt_fpgrowth",
        "borgelt_eclat",
        # PyFIM: Borgelt's C algorithms via CPython extension. Same Python-FFI
        # boundary as fastapriori, so this comparison isolates pure algorithm.
        "pyfim_apriori",
        "pyfim_fpgrowth",
        "pyfim_eclat",
        # pyECLAT (pure Python) and spmf-py (Python wrapper over spmf.jar) --
        # comment out either to skip for a given run.
        #"pyeclat_eclat",
        "spmf_py_fpgrowth",
        "spmf_py_eclat",
        "spmf_fpgrowth",
        "spmf_eclat",
    ],
    3: [
        "fastapriori_fast",
        "fastapriori_classic",
        "borgelt_apriori",
        "borgelt_fpgrowth",
        "borgelt_eclat",
        "pyfim_apriori",
        "pyfim_fpgrowth",
        "pyfim_eclat",
        # "spmf_py_fpgrowth",
        # "spmf_py_eclat",
    ],
    4: [
        "fastapriori_fast",
        "fastapriori_classic",
        "borgelt_apriori",
        "borgelt_fpgrowth",
        "borgelt_eclat",
        "pyfim_apriori",
        "pyfim_fpgrowth",
        "pyfim_eclat",
        # "spmf_py_fpgrowth",
        # "spmf_py_eclat",
    ],
    5: [
        "fastapriori_fast",
        "fastapriori_classic",
        "borgelt_apriori",
        "borgelt_fpgrowth",
        "borgelt_eclat",
        "pyfim_apriori",
        "pyfim_fpgrowth",
        "pyfim_eclat",
        # "spmf_py_fpgrowth",
        # "spmf_py_eclat",
    ],
}

# ---------------------------------------------------------------------------
# Smoke overrides (laptop sanity, 5 min)
# ---------------------------------------------------------------------------
SUPPORT_GRID_SMOKE = {
    "Groceries":     {2: [0.01, 0.02], 3: [0.02]},
    "BMS-WebView-1": {2: [0.005, 0.01]},
}
METHODS_BY_K_SMOKE = {
    2: ["fastapriori_fast", "fastapriori_fast_1t", "fastapriori_classic",
        "efficient_apriori",
        "borgelt_apriori", "borgelt_fpgrowth", "borgelt_eclat",
         "pyfim_apriori",
        "pyfim_fpgrowth",
        "pyfim_eclat",
        # pyECLAT (pure Python) and spmf-py (Python wrapper over spmf.jar) --
        # comment out either to skip for a given run.
        #"pyeclat_eclat",
        "spmf_py_fpgrowth",
        "spmf_py_eclat",
        "spmf_fpgrowth",
        "spmf_eclat",],
    3: ["fastapriori_fast", "borgelt_fpgrowth", "pyfim_fpgrowth",
        "spmf_py_fpgrowth"],
}

# ---------------------------------------------------------------------------
# Resolve active config
# ---------------------------------------------------------------------------
if SMOKE:
    SUPPORT_GRID = SUPPORT_GRID_SMOKE
    METHODS_BY_K = METHODS_BY_K_SMOKE
    br.RUNS_BY_SIZE_CLASS = {"fast": 3, "medium": 3, "heavy": 3}
else:
    SUPPORT_GRID = SUPPORT_GRID_FULL
    METHODS_BY_K = METHODS_BY_K_FULL

# ---------------------------------------------------------------------------
# Run ID: every notebook session gets its own raw/agg CSV so re-runs don't
# stomp on earlier results. Override RUN_ID by hand to resume a prior session.
# ---------------------------------------------------------------------------
from datetime import datetime
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")  # e.g. "20260418_1530"
br.set_raw_csv_suffix(RUN_ID)                       # rebinds br.RAW_CSV / br.AGG_CSV
print(f"Run ID: {RUN_ID}")
print(f"RAW CSV: {br.RAW_CSV}")
print(f"AGG CSV: {br.AGG_CSV}")

# Derived collections (used downstream by plotting / aggregation)
DATASETS = list(SUPPORT_GRID.keys())
K_VALUES = sorted({k for per_k in SUPPORT_GRID.values() for k in per_k.keys()})
METHODS = sorted({m for methods in METHODS_BY_K.values() for m in methods},
                 key=lambda m: br.ALL_METHODS.index(m) if m in br.ALL_METHODS else 99)

CONFIG = {
    "smoke": SMOKE,
    "datasets": DATASETS,
    "k_values": K_VALUES,
    "methods": METHODS,
    "methods_by_k": METHODS_BY_K,
    "support_grid": SUPPORT_GRID,
    "timeout_s": 1500,
    "memout_gb": 60,
    "results_dir": str(br.BENCHMARKS_DIR),
    "raw_csv": str(br.RAW_CSV),
    "agg_csv": str(br.AGG_CSV),
}
memout_kb = CONFIG["memout_gb"] * 1024 * 1024

# ---------------------------------------------------------------------------
# Diagnostics
# ---------------------------------------------------------------------------
total_configs = sum(
    len(supports) * len(METHODS_BY_K.get(k, []))
    for ds, per_k in SUPPORT_GRID.items()
    for k, supports in per_k.items()
)

print(f"Mode: {'SMOKE' if SMOKE else 'FULL'}")
print(f"Datasets ({len(DATASETS)}): {DATASETS}")
print(f"k values present: {K_VALUES}")
print(f"Methods (union across k): {METHODS}")
print()
print("Per-k method count:")
for k in K_VALUES:
    print(f"  k={k}: {len(METHODS_BY_K.get(k, []))} methods -> {METHODS_BY_K.get(k, [])}")
print()
print("Per-(dataset, k) support counts:")
for ds in DATASETS:
    line = ", ".join(f"k={k}: {len(s)}" for k, s in SUPPORT_GRID[ds].items())
    print(f"  {ds}: {line}")
print()
print(f"Total (method, dataset, k, support) configs: {total_configs}")
print(f"Raw CSV: {CONFIG['raw_csv']}")

## Section 1 — Install baselines (idempotent)

Set `INSTALL = True` once; the cell is a no-op on subsequent runs if the binaries/jar are already present.

The cell prints (but does **not** execute) the three OS-tuning commands from `memory_profile.md` §2.B. Run them yourself in a root shell before kicking off the full grid.

In [3]:
INSTALL = True

import subprocess
from pathlib import Path

br.TOOLS_DIR.mkdir(parents=True, exist_ok=True)
br.TOOLS_BIN.mkdir(parents=True, exist_ok=True)
br.TOOLS_SRC.mkdir(parents=True, exist_ok=True)

def _sh(cmd: str, **kwargs):
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, **kwargs)

if INSTALL and platform.system() == "Linux":
    # 1. System tools
    needed_apt = []
    if not Path("/usr/bin/time").exists(): needed_apt.append("time")
    if not shutil.which("gcc"):             needed_apt.append("build-essential")
    if not shutil.which("java"):            needed_apt.append("default-jre-headless")
    if not shutil.which("wget"):            needed_apt.append("wget")
    if needed_apt:
        print(f"Installing system packages: {needed_apt}")
        _sh("sudo -n apt-get update -qq || true")
        _sh(f"sudo -n apt-get install -y {' '.join(needed_apt)}")
    else:
        print("\u2705 System tools already present.")

    # 2. Borgelt
    borgelt_urls = {
        "apriori":  "https://borgelt.net/src/apriori.tar.gz",
        "fpgrowth": "https://borgelt.net/src/fpgrowth.tar.gz",
        "eclat":    "https://borgelt.net/src/eclat.tar.gz",
    }
    for name, url in borgelt_urls.items():
        bin_path = br.TOOLS_BIN / name
        if bin_path.exists():
            print(f"\u2705 borgelt/{name} already built.")
            continue
        print(f"Downloading + building borgelt/{name} ...")
        tar = br.TOOLS_SRC / f"{name}.tar.gz"
        _sh(f"wget -q -O {tar} {url}")
        _sh(f"tar -xzf {tar} -C {br.TOOLS_SRC}")
        # Borgelt's tarballs unpack to <name>/ ; makefile is in <name>/src
        src_dir = br.TOOLS_SRC / name / "src"
        if src_dir.exists():
            r = _sh(f"make -C {src_dir}")
            # The binary lands next to the makefile or in a sibling dir
            candidates = list((br.TOOLS_SRC / name).rglob(name))
            candidates = [c for c in candidates if c.is_file() and os.access(c, os.X_OK)]
            if candidates:
                shutil.copy2(candidates[0], bin_path)
                os.chmod(bin_path, 0o755)
                print(f"  \u2705 {bin_path}")
            else:
                print(f"  \u26a0\ufe0f  {name} built but binary not found; check {src_dir}")
        else:
            print(f"  \u26a0\ufe0f  expected {src_dir}; tarball layout may have changed.")

    # 3. SPMF
    if br.SPMF_JAR.exists():
        print("\u2705 SPMF jar already present.")
    else:
        print("Downloading SPMF ...")
        _sh(f"wget -q -O {br.SPMF_JAR} https://www.philippe-fournier-viger.com/spmf/spmf.jar")

    # 4. Python deps
    try:
        import mlxtend, efficient_apriori  # noqa: F401
        print("\u2705 mlxtend + efficient-apriori installed.")
    except ImportError:
        _sh(f"{sys.executable} -m pip install -q mlxtend efficient-apriori")

    # 4b. PyFIM (Borgelt's C algorithms as a CPython extension). Lets us
    # compare fastapriori against Borgelt across the same Python-FFI
    # boundary, in addition to the subprocess-driven borgelt_* binaries.
    try:
        import fim  # noqa: F401
        print("\u2705 pyfim installed.")
    except ImportError:
        _sh(f"{sys.executable} -m pip install -q pyfim pyECLAT spmf")

    # 5. OS tuning (print only)
    print()
    print("OS tuning (run these yourself in a root shell before the full grid):")
    print("  sudo swapoff -a")
    print("  sudo cpupower frequency-set -g performance")
    print("  sudo sysctl -w vm.drop_caches=3")

tools_after = br.detect_tools()
print()
for k, v in tools_after.items():
    mark = "\u2705" if v else "\u274c"
    print(f"  {mark} {k}")

✅ System tools already present.
✅ borgelt/apriori already built.
✅ borgelt/fpgrowth already built.
✅ borgelt/eclat already built.
✅ SPMF jar already present.
✅ mlxtend + efficient-apriori installed.
✅ pyfim installed.

OS tuning (run these yourself in a root shell before the full grid):
  sudo swapoff -a
  sudo cpupower frequency-set -g performance
  sudo sysctl -w vm.drop_caches=3

  ✅ usr_bin_time
  ✅ gcc
  ✅ make
  ✅ java
  ✅ wget
  ✅ borgelt_apriori
  ✅ borgelt_fpgrowth
  ✅ borgelt_eclat
  ✅ spmf_jar
  ✅ mlxtend
  ✅ efficient_apriori
  ✅ fastapriori
  ✅ pyfim
  ✅ pyeclat
  ✅ spmf_py


## Section 3 — Correctness oracle

Before any timing, confirm every method produces the same set of frequent itemsets on a small dataset. We compare **fastapriori_fast** as the reference against every other method on Groceries at `support=0.01`, `k=2`. Methods that mismatch are logged and excluded from the timing grid.

In [4]:
# Oracle uses itemset counts as the proxy for correctness. Exact-set matching
# would require parsing each tool's output format back into integer itemsets;
# count agreement is a strong necessary-condition check and catches ~all real
# bugs (mis-linked binary, wrong support semantics, off-by-one).

from load_datasets import ALL_DATASETS

ORACLE_DATASET = "Groceries"
ORACLE_K = 2
ORACLE_SUPPORT = 0.01

groceries_df = ALL_DATASETS[ORACLE_DATASET]()
print(f"{ORACLE_DATASET}: {len(groceries_df):,} rows, "
      f"{groceries_df.iloc[:, 0].nunique():,} txns, "
      f"{groceries_df.iloc[:, 1].nunique():,} items")

oracle_rows = []
available_methods = []
for m in METHODS:
    if m.startswith("borgelt_") and not (br.TOOLS_BIN / m.replace("borgelt_", "")).exists():
        oracle_rows.append({"method": m, "status": "skipped", "reason": "binary missing"})
        continue
    if m.startswith("spmf_") and not br.SPMF_JAR.exists():
        oracle_rows.append({"method": m, "status": "skipped", "reason": "spmf.jar missing"})
        continue
    try:
        row = br.run_method(
            m, ORACLE_DATASET, groceries_df, ORACLE_K, ORACLE_SUPPORT,
            timeout_s=120, memout_kb=memout_kb, env_snapshot=env,
        )
        oracle_rows.append({
            "method": m, "status": row.get("status"),
            "n_itemsets": row.get("n_itemsets"),
            "wall_s": row.get("wall_s"),
            "reason": row.get("stderr_tail", "")[:200] if row.get("status") != "ok" else "",
        })
        if row.get("status") == "ok":
            available_methods.append(m)
    except Exception as e:
        oracle_rows.append({"method": m, "status": "error", "reason": repr(e)})

import pandas as pd
oracle_df = pd.DataFrame(oracle_rows)

# Set reference = fastapriori_fast's itemset count
ref_row = oracle_df[oracle_df["method"] == "fastapriori_fast"]
if len(ref_row) > 0 and ref_row.iloc[0]["status"] == "ok":
    ref_n = int(ref_row.iloc[0]["n_itemsets"])
    oracle_df["matches_fast"] = oracle_df["n_itemsets"].apply(
        lambda n: (n == ref_n) if pd.notna(n) else False
    )
else:
    ref_n = None
    oracle_df["matches_fast"] = False

print(f"\nReference (fastapriori_fast) itemset count: {ref_n}")
display_cols = ["method", "status", "n_itemsets", "wall_s", "matches_fast", "reason"]
print(oracle_df[display_cols].to_string(index=False))

# Exclude any method that didn't produce the same count (within a tolerance of 0)
excluded = oracle_df[(oracle_df["status"] != "ok") | (~oracle_df["matches_fast"])]
if len(excluded) > 0:
    lines = [f"# Excluded methods (oracle mismatch on {ORACLE_DATASET})\n"]
    for _, r in excluded.iterrows():
        lines.append(f"- **{r['method']}** — status={r['status']}, "
                     f"n_itemsets={r.get('n_itemsets')}, reason={r.get('reason','')[:200]}")
    br.EXCLUDED_MD.write_text("\n".join(lines))
    print(f"\nWrote exclusion log: {br.EXCLUDED_MD}")

print(f"\nMethods available for timing grid: {available_methods}")

Groceries: 43,367 rows, 9,835 txns, 169 items

Reference (fastapriori_fast) itemset count: 213
             method status  n_itemsets  wall_s  matches_fast reason
   fastapriori_fast     ok         213    0.34          True       
fastapriori_classic     ok         213    0.34          True       
  efficient_apriori     ok         213    0.48          True       
    borgelt_apriori     ok         213    0.00          True       
   borgelt_fpgrowth     ok         213    0.00          True       
      borgelt_eclat     ok         213    0.00          True       
      pyfim_apriori     ok         213    0.42          True       
     pyfim_fpgrowth     ok         213    0.42          True       
        pyfim_eclat     ok         213    0.42          True       
   spmf_py_fpgrowth     ok         213    0.66          True       
      spmf_py_eclat     ok         213    0.63          True       
      spmf_fpgrowth     ok         213    0.38          True       
         spmf_eclat  

## Section 4 — Measurement grid

Per `(method, dataset, k, min_support)`:
1. If all planned `run_id`s are already in `results_rigorous_raw.csv`, skip.
2. Otherwise run a probe (run_id=0, warm-up), classify into fast/medium/heavy, run N more.
3. Append one CSV row per run (not per config).

Safe to Ctrl+C and restart; already-completed rows are detected by `(dataset, method, k, min_support, run_id)`.

In [10]:
import time
from itertools import product

try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(x, **k): return x

# Preload datasets once (file I/O and item-ID remapping are one-time costs)
dataset_frames: dict[str, pd.DataFrame] = {}
for ds in DATASETS:
    try:
        dataset_frames[ds] = ALL_DATASETS[ds]()
        print(f"Loaded {ds}: {len(dataset_frames[ds]):,} rows")
    except Exception as e:
        print(f"⚠️  Skipping {ds}: {e}")

# Build the full task list.
# Iteration order: k → dataset → method → support (desc).
# Why this order:
#   • k first, so easy low-k results land on disk before harder high-k runs.
#   • supports sorted DESCENDING within each (k, dataset, method) group so
#     the cheapest support (e.g. 0.05) runs first and the lowest support
#     (e.g. 0.0001) — which most often hits timeout / oom / huge RSS — runs last.
#     That way, if a run is interrupted or a VM preempted, we have a populated
#     CSV for the "easy" configurations rather than losing everything.

# Per-method support floors: SPMF Java is unreliable below 0.001 on the
# datasets we report -- skip those configurations rather than publish
# timeout/oom rows that add no information.
SPMF_METHODS = {"spmf_fpgrowth", "spmf_eclat", "spmf_py_fpgrowth", "spmf_py_eclat"}
SPMF_MIN_SUPPORT = 0.001

tasks = []
for k in K_VALUES:
    methods_at_k = [m for m in METHODS_BY_K.get(k, []) if m in available_methods]
    if not methods_at_k:
        continue
    for dataset, per_k in SUPPORT_GRID.items():
        if dataset not in dataset_frames:
            continue
        supports = per_k.get(k)
        if not supports:
            continue
        # Highest support first, lowest (hardest) last.
        supports_desc = sorted(supports, reverse=True)
        for method in methods_at_k:
            for support in supports_desc:
                # SPMF (both direct-java and the spmf-py wrapper) becomes
                # unreliable below 0.001: the Java process either runs out of
                # heap or the itemset dump exhausts disk. Skip those configs
                # rather than letting them fail silently as timeout/oom rows.
                if method in SPMF_METHODS and support < SPMF_MIN_SUPPORT:
                    continue
                tasks.append((dataset, method, k, support))

print(f"Total tasks: {len(tasks)}")
if tasks:
    print(f"First task: {tasks[0:5]}")
    print(f"Last task:  {tasks[-1:-5]}")

# Resume: load existing raw CSV
raw_so_far = br.load_raw()
print(f"Existing rows in raw CSV: {len(raw_so_far)}")


Loaded Groceries: 43,367 rows
Loaded BMS-WebView-2: 358,278 rows
Loaded Retail (Belgian): 908,576 rows
Loaded Chainstore: 8,042,879 rows
Loaded Instacart: 32,434,489 rows
Loaded Online Retail: 525,461 rows
Loaded Kosarak: 8,018,988 rows
Loaded BMS-WebView-1: 149,639 rows
Total tasks: 1344
First task: [('Groceries', 'fastapriori_fast', 2, 0.01), ('Groceries', 'fastapriori_fast', 2, 0.005), ('Groceries', 'fastapriori_fast', 2, 0.001), ('Groceries', 'fastapriori_fast', 2, 0.0005), ('Groceries', 'fastapriori_fast', 2, 0.0001)]
Last task:  []
Existing rows in raw CSV: 0


In [ ]:
grid_start = time.monotonic()

for dataset, method, k, support in tqdm(tasks, desc="configs"):
    df = dataset_frames[dataset]
    # Resume support: drop already-completed run_ids
    existing = br.already_done(raw_so_far, dataset, method, k, support)

    # Decide how many we already *need*: at least N+1 (warm-up + N keeps). If
    # we have >= 4 rows already (warm-up + 3), assume config is complete.
    if len(existing) >= 4:
        continue

    try:
        new_rows = br.measure(
            method, dataset, df, k, support,
            timeout_s=CONFIG["timeout_s"],
            memout_kb=memout_kb,
            env_snapshot=env,
            existing_run_ids=existing,
        )
    except Exception as e:
        print(f"\u274c {method}/{dataset}/k={k}/sup={support}: {e!r}")
        continue

    for row in new_rows:
        br.append_raw_row(row)

    if k==2:
        raw_so_far.to_csv('k=2 all_py output.csv', index= False)

    if k==3:
        raw_so_far.to_csv('k=3 all_py output.csv', index= False)

    if k==4:
        raw_so_far.to_csv('k=4 all_py output.csv', index= False)    

    raw_so_far = br.load_raw()  # refresh in-memory view so the next loop sees these

print(f"\nTotal grid wall time: {(time.monotonic() - grid_start) / 60:.1f} min")
print(f"Final raw CSV rows:   {len(raw_so_far)}")

configs:   0%|          | 0/1344 [00:00<?, ?it/s]

In [ ]:
raw = br.load_raw()
print(f"Raw rows: {len(raw):,}")

agg = br.aggregate(raw, reference_method="fastapriori_fast")
br.save_aggregate(agg)
print(f"Aggregated rows: {len(agg):,}")
print(f"Saved to: {br.AGG_CSV}")

# Quick eyeball
preview_cols = [
    "dataset", "method", "k", "min_support",
    "wall_s_median", "wall_s_iqr", "peak_rss_kb_median",
    "n_itemsets_median", "speedup_vs_fastapriori_fast_median",
    "wall_s_n_ok", "wall_s_n_timeout", "wall_s_n_oom",
]
preview_cols = [c for c in preview_cols if c in agg.columns]
agg[preview_cols].head(20)

In [ ]:
# ---- Plots ----
import matplotlib.pyplot as plt
import numpy as np

PLOT_DIR = br.BENCHMARKS_DIR / "images" /
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def _savefig(fig, name):
    fig.savefig(PLOT_DIR / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(PLOT_DIR / f"{name}.png", bbox_inches="tight", dpi=150)
    plt.close(fig)

MARKERS = ["o", "s", "D", "^", "v", ">", "<", "P", "X", "*"]
method_to_marker = {m: MARKERS[i % len(MARKERS)] for i, m in enumerate(METHODS)}

# Pair fastapriori_fast (MT) and fastapriori_fast_1t (1T) on the same panel by
# sharing colour + marker. 1T draws dashed, MT solid. That tells the reader at
# a glance "same library, two configurations; the dashed→solid gap IS the
# parallelism headroom" — which is the paper's core k=2 story.
_PAIRED = {"fastapriori_fast_1t": "fastapriori_fast"}
def _style_for(method: str, default_color):
    if method in _PAIRED:
        return {"color": None, "linestyle": "--", "alpha": 0.85,
                "_pair_with": _PAIRED[method]}
    return {"color": None, "linestyle": "-", "alpha": 0.85, "_pair_with": None}

def _plot_metric_vs_support(metric_median: str, metric_q25: str, metric_q75: str,
                            ylabel: str, fname_prefix: str):
    for dataset in agg["dataset"].dropna().unique():
        for k in sorted(agg["k"].dropna().unique()):
            sub = agg[(agg["dataset"] == dataset) & (agg["k"] == k)]
            if len(sub) == 0:
                continue
            fig, ax = plt.subplots(figsize=(7, 4.5))

            # Pass 1: plot everything EXCEPT paired 1T variants, remembering
            # the auto-assigned colour of each base method so the 1T pass can
            # reuse it.
            color_for_method: dict[str, str] = {}
            for method, g in sub.groupby("method"):
                if method in _PAIRED:
                    continue  # handled in pass 2
                g = g.sort_values("min_support")
                ok = g[g[f"{metric_median[:-7]}_n_ok"] > 0] \
                    if f"{metric_median[:-7]}_n_ok" in g.columns else g
                if len(ok) == 0:
                    continue
                yerr = [ok[metric_median] - ok[metric_q25],
                        ok[metric_q75] - ok[metric_median]]
                (line,) = ax.plot([], [])  # reserve next colour from cycler
                c = line.get_color()
                color_for_method[method] = c
                eb = ax.errorbar(
                    ok["min_support"], ok[metric_median], yerr=yerr,
                    marker=method_to_marker[method], label=method,
                    capsize=2, linewidth=1.2, alpha=0.85,
                    color=c, linestyle="-",
                )

            # Pass 2: paired 1T variants, dashed, same colour + marker as the
            # base method. If the base method has no data for this panel we
            # still give the 1T row its own colour so it doesn't vanish.
            for method, g in sub.groupby("method"):
                if method not in _PAIRED:
                    continue
                g = g.sort_values("min_support")
                ok = g[g[f"{metric_median[:-7]}_n_ok"] > 0] \
                    if f"{metric_median[:-7]}_n_ok" in g.columns else g
                if len(ok) == 0:
                    continue
                yerr = [ok[metric_median] - ok[metric_q25],
                        ok[metric_q75] - ok[metric_median]]
                base = _PAIRED[method]
                c = color_for_method.get(base)
                if c is None:
                    (line,) = ax.plot([], [])
                    c = line.get_color()
                ax.errorbar(
                    ok["min_support"], ok[metric_median], yerr=yerr,
                    marker=method_to_marker.get(base, method_to_marker[method]),
                    label=f"{method} (1 thread)",
                    capsize=2, linewidth=1.2, alpha=0.85,
                    color=c, linestyle="--",
                )

            ax.set_xscale("log")
            ax.set_yscale("log")
            ax.set_xlabel("min_support")
            ax.set_ylabel(ylabel)
            ax.set_title(f"{dataset} — k={int(k)}")
            ax.grid(True, which="both", linestyle=":", alpha=0.4)
            ax.legend(fontsize=7, loc="best")
            _savefig(fig, f"{fname_prefix}_{dataset.replace(' ','_')}_k{int(k)}")

_plot_metric_vs_support("wall_s_median", "wall_s_q25", "wall_s_q75",
                        "wall time (s)", "runtime_vs_support")
_plot_metric_vs_support("peak_rss_kb_median", "peak_rss_kb_q25", "peak_rss_kb_q75",
                        "peak RSS (kB)", "rss_vs_support")
print(f"Saved runtime + RSS plots to {PLOT_DIR}")